# Part 4 in code — benchmarking, ablations, IPS, and what the results actually mean

This is where the framework has to pay rent.

The benchmark is synthetic, but it is not decorative.
It is built to answer the paper’s operational question:
does an explicit slow/fast predictive-state model beat weaker baselines when the data-generating process actually depends on both durable structure and recent interaction state?

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

results_dir = project_root / "results"
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

In [2]:
dataset = pd.read_csv(results_dir / "toy_benchmark_dataset.csv")
main_results = pd.read_csv(results_dir / "benchmark_main_results.csv")
probe_results = pd.read_csv(results_dir / "benchmark_probe_results.csv")
cold_results = pd.read_csv(results_dir / "benchmark_cold_start_results.csv")
person_holdout_results = pd.read_csv(results_dir / "benchmark_person_holdout_results.csv")
ips_results = pd.read_csv(results_dir / "ips_results.csv")

dataset.head()

,person_id,global_time,local_t,regime,y,probe,prop_action,mu_propensity,touch_count,prior_reply_rate,prior_price_count,last_delay,cold_start,w0,w1,c0,c1,c2,x0,x1,x2,T0,T1,Tcol0,Tcol1,...,seq8,seq9,seq10,seq11,seq12,seq13,seq14,seq15,seq16,seq17,seq18,seq19,seq20,seq21,seq22,seq23,seq24,seq25,seq26,seq27,seq28,seq29,u_x_0,u_x_1,u_x_2
0,0,0,0,founder,0,1,0,0.588865,0,0.000000,0,1,1,0.000000,1.000000,1.0,0.0,0.0,1.0,0.2,0.0,0.916391,0.447681,0.916391,0.447681,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,-0.552551,-0.110510,0.000000
1,0,1,1,buyer,0,0,0,0.438166,1,0.000000,0,1,0,0.058790,0.999406,0.0,1.0,0.0,1.0,0.2,0.0,0.942569,0.727708,0.916391,0.447681,...,1.0,0.186617,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,-0.552551,-0.110510,0.000000
2,0,2,2,manager,1,1,2,0.287996,2,0.000000,0,1,0,0.117376,0.997623,0.0,0.0,1.0,0.3,0.4,1.0,0.936869,0.687045,0.916391,0.447681,...,1.0,0.186617,0.0,0.0,1.0,1.0,0.2,0.0,1.160344,0.112029,1.0,0.186617,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,-0.165765,-0.221020,-0.552551
3,0,3,3,founder,0,1,0,0.598687,3,0.333333,1,1,0,0.175556,0.994654,1.0,0.0,0.0,1.0,0.2,0.0,0.988768,0.166508,0.916391,0.447681,...,1.0,0.936003,0.0,0.0,0.0,1.0,0.2,0.0,1.160344,0.112029,1.0,0.186617,0.0,0.0,1.0,1.0,0.2,0.0,1.160344,0.112029,1.0,0.186617,-0.552551,-0.110510,0.000000
4,0,4,4,buyer,1,0,1,0.465698,4,0.250000,1,1,0,0.233129,0.990503,0.0,1.0,0.0,0.1,1.0,0.2,0.988768,0.166508,0.916391,0.447681,...,1.0,0.186617,2.0,1.0,1.0,0.3,0.4,1.0,0.931915,0.385398,1.0,0.936003,0.0,0.0,0.0,1.0,0.2,0.0,1.160344,0.112029,1.0,0.186617,-0.055255,-0.552551,-0.110510


## Temporal main-task benchmark

In [3]:
main_results

,model,target,logloss,brier,pr_auc,auc
0,latent_collapsed_slow,y,0.625269,0.218164,0.654465,0.702990
1,latent_uniform_pool,y,0.629743,0.220234,0.649758,0.695217
2,latent_full,y,0.630145,0.220422,0.648903,0.694166
3,latent_no_slow,y,0.638034,0.223760,0.651785,0.683451
4,latent_no_fast,y,0.643151,0.226271,0.642520,0.670075
5,static,y,0.660353,0.234351,0.591304,0.651236
6,current_touch,y,0.662452,0.234791,0.574402,0.632257
7,two_tower_like,y,0.668185,0.234230,0.586359,0.666538
8,shallow_history,y,0.680713,0.242733,0.574341,0.634218
9,monolithic_sequence,y,0.684939,0.242663,0.593755,0.633658


## Probe benchmark

In [4]:
probe_results

,model,target,logloss,brier,pr_auc,auc
0,latent_no_slow,probe,0.586499,0.199318,0.780688,0.671336
1,latent_full,probe,0.590217,0.201178,0.785102,0.668645
2,latent_uniform_pool,probe,0.590315,0.201230,0.783986,0.667458
3,latent_collapsed_slow,probe,0.591891,0.201702,0.778306,0.659861
4,current_touch,probe,0.626966,0.217945,0.724096,0.577279
5,latent_no_fast,probe,0.629999,0.218759,0.712945,0.570513
6,frequency,probe,0.630599,0.219383,0.675000,0.500000
7,monolithic_sequence,probe,0.651877,0.227998,0.769994,0.610715
8,shallow_history,probe,0.656006,0.228331,0.739376,0.589744
9,static,probe,0.678700,0.239166,0.700140,0.537274


## Cold-start slices

The first one is the temporal cold-start slice from the temporally held-out set.

The second is a stricter person-holdout first-touch slice.

In [5]:
cold_results, person_holdout_results

(                    model target   logloss     brier    pr_auc       auc
 0               frequency      y  0.690061  0.248457  0.458333  0.500000
 1   latent_collapsed_slow      y  0.710446  0.257009  0.615995  0.517483
 2          two_tower_like      y  0.714275  0.256971  0.534373  0.601399
 3     latent_uniform_pool      y  0.717902  0.260444  0.627305  0.531469
 4             latent_full      y  0.718584  0.260705  0.627305  0.531469
 5           current_touch      y  0.720479  0.263139  0.529111  0.468531
 6          latent_no_fast      y  0.741124  0.270031  0.568666  0.496503
 7          latent_no_slow      y  0.755946  0.278699  0.581054  0.475524
 8                  static      y  0.777462  0.283861  0.478235  0.496503
 9         shallow_history      y  0.788795  0.286406  0.485253  0.510490
 10    monolithic_sequence      y  0.790822  0.287723  0.528485  0.517483,
                     model target   logloss     brier    pr_auc       auc
 0          two_tower_like      y  0.

## IPS sanity check

In [6]:
ips_results

,n,ips_estimate,true_value,naive_logged_mean
0,200000,0.611563,0.609472,0.576827


## A compact interpretation table

In [7]:
def metric(df, model, col):
    return float(df.loc[df.model == model, col].iloc[0])

interpretation = pd.DataFrame([
    {
        "claim": "Full latent model beats static baseline on main task",
        "value": metric(main_results, "static", "logloss") - metric(main_results, "latent_full", "logloss"),
        "unit": "logloss improvement",
    },
    {
        "claim": "Fast state matters on main task",
        "value": metric(main_results, "latent_no_fast", "logloss") - metric(main_results, "latent_full", "logloss"),
        "unit": "logloss improvement",
    },
    {
        "claim": "Slow state matters on main task",
        "value": metric(main_results, "latent_no_slow", "logloss") - metric(main_results, "latent_full", "logloss"),
        "unit": "logloss improvement",
    },
    {
        "claim": "Full latent beats monolithic sequence baseline",
        "value": metric(main_results, "monolithic_sequence", "logloss") - metric(main_results, "latent_full", "logloss"),
        "unit": "logloss improvement",
    },
    {
        "claim": "IPS estimate tracks true value",
        "value": float(abs(ips_results.loc[0, "ips_estimate"] - ips_results.loc[0, "true_value"])),
        "unit": "absolute error",
    },
])
interpretation

,claim,value,unit
0,Full latent model beats static baseline on main task,0.030209,logloss improvement
1,Fast state matters on main task,0.013006,logloss improvement
2,Slow state matters on main task,0.007890,logloss improvement
3,Full latent beats monolithic sequence baseline,0.054794,logloss improvement
4,IPS estimate tracks true value,0.002091,absolute error


## What this means operationally

A sober reading of the toy results is:

- **Yes, explicit state can buy signal.**  
  The full latent-state model beats the weaker baselines on the main target.

- **Yes, the fast state matters.**  
  Removing it hurts.

- **Yes, the slow state matters.**  
  Removing it hurts too.

- **No, you should not canonize every architectural flourish.**  
  On this toy run, collapsed slow pooling was nearly tied with the fuller version. Good. That is exactly why the ablation exists.

- **Yes, the IPS piece matters if you want to claim policy value.**  
  Without logged propensities, you have ranking and simulation. With propensities, you at least have a principled offline estimate.

## Real-world applicability

The project structure is already in the right shape for GTM, support, retention, recruiting, or any domain where:
- the entity has durable traits,
- the local state changes with interaction history,
- propositions are admissible actions/messages/offers,
- and downstream observables are logged.

The adaptation guide in the next notebook turns that into a concrete checklist.